In [1]:
import pandas as pd
import numpy as np

from sklearn.mixture import GaussianMixture
from sklearn.cluster import DBSCAN

In [2]:
DATA_PATH = "../data/processed/normalized_features.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,0,1,2,3,4,5,6,7,8,9,...,30,31,32,33,34,35,36,37,38,file_name
0,-1.063756,0.165255,0.691232,0.246091,0.834620,0.754703,0.890069,0.424851,0.780238,0.928137,...,-0.024551,0.035300,0.032998,0.030189,0.017152,0.025025,0.041286,0.090386,0.122804,03-01-01-01-01-01-01.wav
1,-0.994626,0.277280,0.510977,0.358848,1.173437,0.858311,0.704162,0.687790,0.734243,0.446494,...,-0.024551,0.035301,0.032998,0.030190,0.017150,0.025026,0.041285,0.090387,0.122805,03-01-01-01-01-02-01.wav
2,-0.954325,0.242716,0.744771,0.211534,0.967347,0.384805,0.907501,0.695574,0.545731,0.673504,...,-1.903202,-1.793683,-1.830874,-1.792491,-1.642013,-1.670757,-1.592613,-1.465834,-1.391754,03-01-01-01-02-01-01.wav
3,-0.915118,0.099359,0.902350,0.372147,0.933726,0.609970,1.105841,0.915741,0.450237,0.488136,...,-1.676941,-1.496043,-1.556583,-1.845502,-2.018470,-2.301329,-2.252764,-1.681633,-1.013032,03-01-01-01-02-02-01.wav
4,-1.392440,0.653942,0.922824,0.535235,0.972075,0.888982,0.886653,0.716942,0.400051,1.125225,...,-2.996762,-2.778744,-2.827963,-2.797626,-2.522053,-2.391194,-2.025809,-1.575707,-1.233482,03-01-02-01-01-01-01.wav


In [3]:
file_names = df["file_name"]

X = df.drop(columns=["file_name"])

print("Feature Matrix Shape:", X.shape)

Feature Matrix Shape: (1440, 39)


In [4]:
gmm = GaussianMixture(
    n_components=8,
    covariance_type="full",
    random_state=42
)

gmm_labels = gmm.fit_predict(X)

df["GMM_cluster"] = gmm_labels

In [9]:
from sklearn.metrics import silhouette_score

In [10]:
gmm_score = silhouette_score(X, gmm_labels)
print("GMM Silhouette Score:", gmm_score)

GMM Silhouette Score: 0.024799263286118443


In [5]:
df["GMM_cluster"].value_counts()

GMM_cluster
1    669
0    383
4    125
5     87
7     61
2     47
3     35
6     33
Name: count, dtype: int64

In [6]:
dbscan = DBSCAN(
    eps=2.5,
    min_samples=10
)

dbscan_labels = dbscan.fit_predict(X)

df["DBSCAN_cluster"] = dbscan_labels

In [7]:
df["DBSCAN_cluster"].value_counts()

DBSCAN_cluster
-1    799
 0    641
Name: count, dtype: int64

In [8]:
OUTPUT_PATH = "../results/clustering_results.csv"

df.to_csv(OUTPUT_PATH, index=False)

print("Clustering results saved successfully!")

Clustering results saved successfully!


## Clustering Algorithms

In this notebook we apply unsupervised clustering algorithms to group speech samples based on acoustic features.

### Gaussian Mixture Model (GMM)

GMM assumes that the dataset is generated from a mixture of several Gaussian distributions. Each cluster corresponds to one Gaussian component.

Advantages:
- Flexible cluster shapes
- Probabilistic clustering
- Works well for speech feature distributions

In this project we use **8 components**, corresponding to the 8 emotional categories present in the dataset.

---

### DBSCAN (Density-Based Spatial Clustering)

DBSCAN groups points based on **density of neighboring points**.

Key properties:

- Does not require specifying the number of clusters
- Can detect **noise points**
- Works well when clusters have irregular shapes

Points labeled **-1** are considered noise or outliers.

---

Clustering helps identify **natural groupings in speech features**, which may correspond to emotional patterns in the audio recordings.

### Silhouette Score Evaluation

The Silhouette Score measures how well each data point fits within its assigned cluster compared to other clusters.

The score ranges from **-1 to 1**:

- **1** → well-separated clusters  
- **0** → overlapping clusters  
- **-1** → incorrect clustering

Due to the overlapping acoustic characteristics of emotional speech, moderate silhouette scores are expected.

### Silhouette Score Evaluation

The Silhouette Score obtained for the Gaussian Mixture Model clustering was **0.0248**.

The Silhouette Score measures how well each data point fits within its assigned cluster compared to other clusters.

The score ranges from **-1 to 1**, where higher values indicate better cluster separation.

In this case, the low score suggests that the clusters overlap in the feature space. This behavior is expected because emotional speech signals often share similar acoustic properties. Features such as MFCC capture spectral characteristics of speech, but different emotions may still produce similar patterns.

Therefore, moderate or low silhouette scores are common when clustering speech emotion datasets.